In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 넘겨준 npy 파일 로드
X_raw = np.load('X_2018.npy')
y = np.load('y_2018.npy')

print(f"전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

# 2. Train / Test 분할 (8:2 비율, 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

전체 데이터 형태: X=(200000, 62), y=(200000,)


In [3]:
from sklearn.preprocessing import RobustScaler

# 네트워크 데이터의 극단적인 이상치(Outlier)에 강한 RobustScaler 사용
scaler = RobustScaler()

# X_train에만 fit(기준 설정)과 transform(변환)을 동시에 적용
X_train_scaled = scaler.fit_transform(X_train)

# X_test는 절대 fit하지 않고, Train의 기준에 맞춰 transform(변환)만 적용!
X_test_scaled = scaler.transform(X_test)


In [4]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 과적합 방지(Pruning)가 적용된 랜덤 포레스트 정의
# 기존처럼 무제한으로 트리가 깊어지는 것을 막아 노이즈 암기를 차단합니다.
rf_model_regularized = RandomForestClassifier(
    n_estimators=100,
    max_depth=9,             # 15보단 얕고 8보단 깊은 황금 밸런스
    min_samples_split=15,     # 너무 디테일한 노이즈 분할 완벽 차단
    min_samples_leaf=5,       # 끝단 평탄화 유지
    max_features='sqrt',
    random_state=42
)

# 5. 모델 학습
rf_model_regularized.fit(X_train_scaled, y_train)

THRESHOLD = 0.5

# predict() 대신 predict_proba()를 사용하여 1(공격) 클래스에 대한 확률값만 추출
train_pred_prob = rf_model_regularized.predict_proba(X_train_scaled)[:, 1]
test_pred_prob = rf_model_regularized.predict_proba(X_test_scaled)[:, 1]

# 설정한 THRESHOLD(0.5)를 기준으로 0과 1로 변환
train_pred_strict = (train_pred_prob >= THRESHOLD).astype(int)
test_pred_strict = (test_pred_prob >= THRESHOLD).astype(int)

# 변환된 예측값으로 F1-Score 계산
train_f1 = f1_score(y_train, train_pred_strict, average='weighted')
test_f1 = f1_score(y_test, test_pred_strict, average='weighted')

print(f"- 훈련 데이터 F1-Score : {train_f1:.6f}")
print(f"- 테스트 데이터 F1-Score: {test_f1:.6f}")
print(f"- 점수 격차(Gap)       : {abs(train_f1 - test_f1):.6f}")

if abs(train_f1 - test_f1) < 0.01:
    print("👉 진단: 훈련/테스트 격차가 1% 미만으로 극히 안정적입니다. (과적합 없음)")
else:
    print("👉 진단: 점수 격차가 큽니다. 모델이 훈련 데이터에 과적합되었을 수 있습니다.")

# 7. 최종 성능 및 오탐률(FPR) 평가
acc = accuracy_score(y_test, test_pred_strict)
prec = precision_score(y_test, test_pred_strict, average='weighted')
rec = recall_score(y_test, test_pred_strict, average='weighted')

# 혼동 행렬 추출
cm = confusion_matrix(y_test, test_pred_strict)
tn, fp, fn, tp = cm.ravel()

# 오탐률 계산 (정상인데 공격으로 탐지한 비율)
fpr = fp / (fp + tn)

print("\n[최종 방어 모델 성능]")
print(f"- Accuracy : {acc:.4f}")
print(f"- Precision: {prec:.4f}")
print(f"- Recall   : {rec:.4f}")
print(f"- F1-Score : {test_f1:.4f}")
print(f"🚨 오탐(False Positive) 건수: {fp}건")
print(f"🚨 오탐률(FPR): {fpr*100:.4f}%")

- 훈련 데이터 F1-Score : 0.997994
- 테스트 데이터 F1-Score: 0.997825
- 점수 격차(Gap)       : 0.000169
👉 진단: 훈련/테스트 격차가 1% 미만으로 극히 안정적입니다. (과적합 없음)

[최종 방어 모델 성능]
- Accuracy : 0.9978
- Precision: 0.9978
- Recall   : 0.9978
- F1-Score : 0.9978
🚨 오탐(False Positive) 건수: 78건
🚨 오탐률(FPR): 0.3900%


In [ ]:
import joblib
# 8. 최종 배포용 파일 저장
joblib.dump(rf_model_regularized, 'ids_rf_model_2018.joblib')
joblib.dump(scaler, 'ids_robust_scaler_2018.joblib')
print("\n저장 완료")